# 🧠 EEG Locomotion Decoding — Master Research Notebook
### Artifact-Aware Cross-Subject EEG Locomotion Decoding: Extended Experiments
**Authors:** Md. Mahfujul Karim Sheikh & Md. Shahriar Sajid  
**Dataset:** OpenNeuro ds007526 (N=144, 34,904 epochs)  
**Platform:** Kaggle — Run All & Save ✅

---
### 📋 What this Master Notebook Runs (in one click):
1. **Environment Setup & Data Loading**
2. **Preprocessing Pipeline** — filter, epoch, PSD features
3. **Baseline Benchmark** — RF, XGBoost, LinearSVC
4. **🆕 Track 1: PD Severity Stratification** — Mild / Moderate / Severe PD vs HC
5. **🆕 Track 2: Frequency Band Ablation** — which band matters most?
6. **🆕 Track 3: Transfer Learning** — few-shot subject adaptation
7. **🆕 Track 4: Freezing of Gait (FoG) Detection** — PD clinical task
8. **Final Summary** — all results in one table + CSV export

## ⚙️ Section 1: Environment Setup

In [ ]:
!pip install -q mne pyriemann xgboost seaborn scikit-learn matplotlib pandas numpy

import os, glob, time, gc, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.signal import welch

import mne
from mne.decoding import CSP

from sklearn.model_selection import GroupKFold
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')
print("All libraries loaded successfully")

## 📂 Section 2: Dataset Configuration & Auto-Detection

In [ ]:
# ── AUTO-DETECT DATASET PATH ─────────────────────────────────────────────────
import os, glob

print("=== Scanning /kaggle/input for Dataset ===")
for d in sorted(os.listdir('/kaggle/input')):
    print(f"  Directory: /kaggle/input/{d}")

# 1. Search for participants.tsv (BIDS root identifier)
bids_root = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'participants.tsv' in files:
        bids_root = root
        break

if bids_root:
    DATASET_PATH = bids_root
    print(f"\nFound BIDS root: {DATASET_PATH}")
else:
    set_candidates = sorted(glob.glob('/kaggle/input/**/*.set', recursive=True))
    if set_candidates:
        p = set_candidates[0].replace('\\', '/')
        parts = p.split('/')
        if 'eeg' in parts:
            idx = parts.index('eeg')
            DATASET_PATH = '/'.join(parts[:idx-1])
        else:
            DATASET_PATH = '/'.join(parts[:-2])
        print(f"\nFound .set files. Inferred dataset root: {DATASET_PATH}")
    else:
        DATASET_PATH = '/kaggle/input/ds007526-1-0-1/ds007526-1.0.1'
        print(f"\nWarning: No files found! Defaulting path to: {DATASET_PATH}")

# Search for all EEG .set files
eeg_files = sorted(glob.glob(os.path.join(DATASET_PATH, '**', '*.set'), recursive=True))
if not eeg_files:
    eeg_files = sorted(glob.glob('/kaggle/input/**/*.set', recursive=True))

print(f"Total .set files found: {len(eeg_files)}")
if eeg_files:
    print(f"Example file: {eeg_files[0]}")
else:
    print("WARNING: 0 .set files found! Make sure dataset ds007526-1-0-1 is attached.")

# ── HYPERPARAMETERS ──────────────────────────────────────────────────────────
SFREQ       = 250
EPOCH_LEN   = 2.0
N_SAMPLES   = int(SFREQ * EPOCH_LEN)   # 500 samples
N_FOLDS     = 5
BANDPASS_LO = 1.0
BANDPASS_HI = 40.0
BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'beta':  (13.0, 30.0),
    'gamma': (30.0, 45.0),
}
N_BANDS = len(BANDS)
print(f"Config ready: {N_FOLDS}-Fold GroupKFold | Epoch={EPOCH_LEN}s | Bands={N_BANDS}")

## 🔧 Section 3: Preprocessing & PSD Feature Extraction

In [ ]:
def compute_psd_features(epoch_data, sfreq=250, bands=BANDS):
    n_channels = epoch_data.shape[0]
    features = []
    for ch in range(n_channels):
        freqs, psd = welch(epoch_data[ch], fs=sfreq, nperseg=min(sfreq, epoch_data.shape[1]))
        for band, (lo, hi) in bands.items():
            mask = (freqs >= lo) & (freqs <= hi)
            features.append(float(np.mean(psd[mask])) if mask.any() else 0.0)
    return np.array(features, dtype=np.float32)


def load_and_preprocess(file_path, label, subject_id):
    records = []
    try:
        raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
        drop_ch = [ch for ch in raw.ch_names
                   if ch in ['VREF', 'STI 014'] or 'EOG' in ch.upper()]
        raw.drop_channels([c for c in drop_ch if c in raw.ch_names])
        raw.set_montage('standard_1020', on_missing='ignore', verbose=False)
        raw.filter(BANDPASS_LO, BANDPASS_HI, fir_design='firwin', verbose=False)
        raw.notch_filter(50, verbose=False)
        data = raw.get_data()
        n_epochs = data.shape[1] // N_SAMPLES
        for i in range(n_epochs):
            seg = data[:, i*N_SAMPLES:(i+1)*N_SAMPLES]
            if seg.shape[1] == N_SAMPLES:
                records.append({
                    'features': compute_psd_features(seg),
                    'label': label,
                    'subject': subject_id
                })
        del raw, data
        gc.collect()
    except Exception as e:
        pass
    return records


def load_full_dataset(file_list):
    all_records = []
    total = len(file_list)
    print(f"Processing {total} EEG recordings...")
    start_t = time.time()
    for idx, fpath in enumerate(file_list):
        parts = fpath.replace('\\', '/').split('/')
        subj_id = next((p for p in parts if p.startswith('sub-')), fpath)
        label = 1 if 'task-walk' in fpath else (0 if 'task-rest' in fpath else -1)
        if label == -1:
            continue
        recs = load_and_preprocess(fpath, label, subj_id)
        all_records.extend(recs)
        if (idx + 1) % 25 == 0 or (idx + 1) == total:
            elapsed = time.time() - start_t
            print(f"  [{idx+1}/{total}] files processed ({len(all_records)} epochs loaded, {elapsed:.1f}s)")

    if not all_records:
        raise RuntimeError("No epochs loaded! Check that the dataset ds007526-1-0-1 is added to this notebook.")

    X = np.array([r['features'] for r in all_records], dtype=np.float32)
    y = np.array([r['label']    for r in all_records], dtype=np.int32)
    groups_raw = [r['subject']  for r in all_records]
    unique_subjects = sorted(set(groups_raw))
    subj_map = {s: i for i, s in enumerate(unique_subjects)}
    groups = np.array([subj_map[s] for s in groups_raw], dtype=np.int32)
    print(f"\nFinal Dataset: {len(X)} epochs | {len(unique_subjects)} subjects | "
          f"Rest={int(np.sum(y==0))}, Walk={int(np.sum(y==1))} | "
          f"Features/epoch={X.shape[1]}")
    return X, y, groups, unique_subjects

print("Loading dataset...")
X, y, groups, subjects = load_full_dataset(eeg_files)

## 📊 Section 4: Baseline Model Benchmark

In [ ]:
def run_crossval(model, X, y, groups, model_name, n_folds=5):
    gkf = GroupKFold(n_splits=n_folds)
    fold_acc, fold_bacc, fold_f1, fold_auc = [], [], [], []

    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = (model.predict_proba(X_test)[:, 1]
                  if hasattr(model, 'predict_proba') else y_pred.astype(float))

        fold_acc.append(accuracy_score(y_test, y_pred) * 100)
        fold_bacc.append(balanced_accuracy_score(y_test, y_pred) * 100)
        fold_f1.append(f1_score(y_test, y_pred, average='macro') * 100)
        try:
            fold_auc.append(roc_auc_score(y_test, y_prob) * 100)
        except Exception:
            fold_auc.append(0.0)

    def fmt(vals):
        return f"{np.mean(vals):.2f} +/- {np.std(vals):.2f}"

    return {
        'Model':        model_name,
        'Accuracy':     fmt(fold_acc),
        'Balanced Acc': fmt(fold_bacc),
        'F1 Score':     fmt(fold_f1),
        'ROC-AUC':      fmt(fold_auc),
        'acc_vals':     fold_acc,
        'acc_mean':     float(np.mean(fold_acc)),
    }


models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':       XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6,
                                   subsample=0.8, colsample_bytree=0.8,
                                   eval_metric='logloss', random_state=42, n_jobs=-1),
    'LinearSVC':     LinearSVC(C=1.0, max_iter=5000, random_state=42),
}

baseline_results = []
for name, model in models.items():
    print(f"Training {name}...")
    res = run_crossval(model, X, y, groups, name)
    baseline_results.append(res)
    print(f"  Accuracy: {res['Accuracy']}%")

df_baseline = pd.DataFrame([{k: v for k, v in r.items()
                              if k not in ('acc_vals', 'acc_mean')}
                             for r in baseline_results])
print("\nBASELINE BENCHMARK RESULTS:")
print(df_baseline.to_string(index=False))

## 🆕 Track 1: PD Severity Stratification
Classify participants by UPDRS motor score into Mild / Moderate / Severe PD groups
and compare locomotion decoding accuracy.

In [ ]:
participants_tsv = os.path.join(DATASET_PATH, 'participants.tsv')
if not os.path.exists(participants_tsv):
    tsv_matches = glob.glob('/kaggle/input/**/participants.tsv', recursive=True)
    if tsv_matches:
        participants_tsv = tsv_matches[0]

if os.path.exists(participants_tsv):
    df_part = pd.read_csv(participants_tsv, sep='\t')
    print("Participants file loaded. Columns:", df_part.columns.tolist())
else:
    print("participants.tsv not found — using synthetic UPDRS demo values")
    n_subj = len(subjects)
    np.random.seed(42)
    updrs_vals = [1] * 28 + list(np.random.randint(5, 46, max(0, n_subj - 28)))
    df_part = pd.DataFrame({'participant_id': subjects,
                            'updrs_part_iii': updrs_vals[:n_subj],
                            'group': ['HC']*28 + ['PD']*(n_subj-28)})


def get_severity(updrs, grp_label='PD'):
    if grp_label == 'HC':
        return 'HC'
    try:
        u = float(updrs)
    except (ValueError, TypeError):
        return 'Moderate PD'
    if pd.isna(u) or u <= 5:
        return 'HC' if grp_label == 'HC' else 'Mild PD'
    elif u <= 15:
        return 'Mild PD'
    elif u <= 30:
        return 'Moderate PD'
    else:
        return 'Severe PD'


subj_to_group = {}
for _, row in df_part.iterrows():
    pid = str(row.get('participant_id', row.iloc[0]))
    grp = str(row.get('group', 'PD')).strip().upper()
    updrs = None
    for col in ['updrs_part_iii', 'UPDRS_III', 'updrs_iii', 'UPDRS', 'updrs', 'updrs_total', 'motor_score']:
        if col in row.index and pd.notna(row[col]):
            updrs = row[col]
            break
    if grp == 'HC':
        subj_to_group[pid] = 'HC'
    else:
        subj_to_group[pid] = get_severity(updrs, grp_label='PD')

severity_order   = ['HC', 'Mild PD', 'Moderate PD', 'Severe PD']
severity_results = []

for grp in severity_order:
    grp_subj_idx = [i for i, s in enumerate(subjects)
                    if subj_to_group.get(s, 'Unknown') == grp]
    if len(grp_subj_idx) < 3:
        print(f"  {grp}: only {len(grp_subj_idx)} subjects — skipping")
        continue
    mask = np.isin(groups, grp_subj_idx)
    X_g, y_g, g_g = X[mask], y[mask], groups[mask]
    n_folds_g = min(N_FOLDS, len(grp_subj_idx))
    print(f"Training {grp} ({len(grp_subj_idx)} subjects, {len(X_g)} epochs)...")
    res = run_crossval(
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        X_g, y_g, g_g, grp, n_folds=n_folds_g)
    res['Group']      = grp
    res['N_Subjects'] = len(grp_subj_idx)
    severity_results.append(res)

df_severity = pd.DataFrame([{k: v for k, v in r.items()
                              if k not in ('acc_vals', 'acc_mean')}
                             for r in severity_results])
print("\nPD SEVERITY STRATIFICATION RESULTS:")
if not df_severity.empty:
    print(df_severity[['Group', 'N_Subjects', 'Accuracy', 'F1 Score']].to_string(index=False))

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
grp_names = [r['Group']    for r in severity_results]
grp_accs  = [r['acc_mean'] for r in severity_results]
grp_stds  = [float(r['Accuracy'].split('+/-')[1]) for r in severity_results]
colors    = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c'][:len(grp_names)]

bars = ax.bar(grp_names, grp_accs, yerr=grp_stds, capsize=6,
              color=colors, edgecolor='white', linewidth=1.2,
              error_kw={'elinewidth': 2, 'capthick': 2})
for bar, acc in zip(bars, grp_accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_ylabel('Classification Accuracy (%)', fontsize=12)
ax.set_title('Locomotion Decoding Accuracy by PD Severity (Random Forest)',
             fontsize=13, fontweight='bold')
ax.set_ylim(50, 100)
ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Chance level')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/fig_severity_stratification.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: fig_severity_stratification.png")

## 🆕 Track 2: Frequency Band Ablation
Remove one EEG frequency band at a time and measure accuracy drop.
Also test single-band-only classification to find the most informative band.

In [ ]:
BAND_NAMES = list(BANDS.keys())
n_ch = X.shape[1] // N_BANDS  # number of channels

def band_col_indices(band_idx, n_ch=n_ch, n_bands=N_BANDS):
    return [ch * n_bands + band_idx for ch in range(n_ch)]

ablation_results = []
rf_ab = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Full 5-band baseline
print("Full 5-band baseline...")
res_full = run_crossval(rf_ab, X, y, groups, 'All 5 Bands (Baseline)')
ablation_results.append(res_full)
print(f"  {res_full['Accuracy']}%")

# Remove one band at a time
for b_idx, b_name in enumerate(BAND_NAMES):
    keep = [c for c in range(X.shape[1]) if c not in band_col_indices(b_idx)]
    X_ab = X[:, keep]
    label = f'Remove {b_name.capitalize()}'
    print(f"Training '{label}' ({X_ab.shape[1]} features)...")
    res = run_crossval(rf_ab, X_ab, y, groups, label)
    ablation_results.append(res)
    print(f"  {res['Accuracy']}%")

# Single band only
for b_idx, b_name in enumerate(BAND_NAMES):
    only = band_col_indices(b_idx)
    X_single = X[:, only]
    label = f'Only {b_name.capitalize()}'
    print(f"Training '{label}' ({X_single.shape[1]} features)...")
    res = run_crossval(rf_ab, X_single, y, groups, label)
    ablation_results.append(res)
    print(f"  {res['Accuracy']}%")

df_ablation = pd.DataFrame([{'Condition': r['Model'], 'Accuracy': r['Accuracy'],
                              'acc_mean': r['acc_mean']} for r in ablation_results])
print("\nBAND ABLATION RESULTS:")
print(df_ablation[['Condition', 'Accuracy']].to_string(index=False))

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
baseline_acc = ablation_results[0]['acc_mean']

# Left: remove-one
remove_res = ablation_results[:N_BANDS + 1]
r_labels = [r['Model'] for r in remove_res]
r_accs   = [r['acc_mean'] for r in remove_res]
clrs_r   = ['#3498db'] + ['#e74c3c' if baseline_acc - a > 1 else '#95a5a6'
                           for a in r_accs[1:]]
axes[0].barh(r_labels[::-1], r_accs[::-1], color=clrs_r[::-1], edgecolor='white')
axes[0].axvline(baseline_acc, color='navy', linestyle='--', linewidth=2,
                label=f'Baseline {baseline_acc:.1f}%')
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Remove-One Band Ablation', fontweight='bold')
axes[0].legend(); axes[0].set_xlim(50, 100)

# Right: single-band
single_res = ablation_results[N_BANDS + 1:]
s_labels   = [r['Model'] for r in single_res]
s_accs     = [r['acc_mean'] for r in single_res]
clrs_s     = ['#9b59b6', '#1abc9c', '#f39c12', '#e74c3c', '#3498db']
axes[1].bar(s_labels, s_accs, color=clrs_s, edgecolor='white')
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.5, label='Chance')
for i, (lbl, acc) in enumerate(zip(s_labels, s_accs)):
    axes[1].text(i, acc + 0.8, f'{acc:.1f}%', ha='center', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Single-Band Only Classification', fontweight='bold')
axes[1].legend(); axes[1].set_ylim(45, 100)

plt.suptitle('Frequency Band Ablation Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/fig_band_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: fig_band_ablation.png")

## 🆕 Track 3: Transfer Learning — Few-Shot Subject Adaptation
Train on source subjects, then adapt to unseen target subjects using 0%, 5%, 10%, 20%
of their labeled data.

In [ ]:
adapt_ratios = [0.0, 0.05, 0.10, 0.20]
unique_subjs  = np.unique(groups)
gkf_tl        = GroupKFold(n_splits=N_FOLDS)
fold_results_per_ratio = {r: [] for r in adapt_ratios}

for fold_idx, (tr_idx, te_idx) in enumerate(
        gkf_tl.split(unique_subjs, unique_subjs, unique_subjs)):
    train_subjs = unique_subjs[tr_idx]
    test_subjs  = unique_subjs[te_idx]

    train_mask = np.isin(groups, train_subjs)
    X_src, y_src = X[train_mask], y[train_mask]

    scaler_tl = StandardScaler().fit(X_src)
    X_src_sc  = scaler_tl.transform(X_src)

    base_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    base_rf.fit(X_src_sc, y_src)

    for ratio in adapt_ratios:
        fold_accs = []
        for ts in test_subjs:
            ts_mask  = (groups == ts)
            X_ts_raw = X[ts_mask]
            y_ts_raw = y[ts_mask]
            if len(X_ts_raw) < 4:
                continue
            X_ts_sc = scaler_tl.transform(X_ts_raw)

            if ratio == 0.0:
                y_pred   = base_rf.predict(X_ts_sc)
                y_true   = y_ts_raw
            else:
                n_adapt  = max(2, int(len(X_ts_raw) * ratio))
                adapt_idx = np.random.choice(len(X_ts_raw), n_adapt, replace=False)
                test_idx2 = np.array([i for i in range(len(X_ts_raw))
                                      if i not in adapt_idx])
                if len(test_idx2) < 2:
                    continue
                X_aug    = np.vstack([X_src_sc, X_ts_sc[adapt_idx]])
                y_aug    = np.concatenate([y_src, y_ts_raw[adapt_idx]])
                adapted  = clone(base_rf)
                adapted.fit(X_aug, y_aug)
                y_pred   = adapted.predict(X_ts_sc[test_idx2])
                y_true   = y_ts_raw[test_idx2]

            fold_accs.append(accuracy_score(y_true, y_pred) * 100)

        if fold_accs:
            fold_results_per_ratio[ratio].append(float(np.mean(fold_accs)))

    print(f"  Fold {fold_idx+1}/{N_FOLDS} done")

print("\nTRANSFER LEARNING RESULTS:")
tl_summary = []
for ratio in adapt_ratios:
    vals  = fold_results_per_ratio[ratio]
    mean_ = float(np.mean(vals))
    std_  = float(np.std(vals))
    label = 'Zero-shot' if ratio == 0.0 else f'{int(ratio*100)}% target data'
    print(f"  {label:<22}: {mean_:.2f} +/- {std_:.2f}%")
    tl_summary.append({'label': label, 'mean': mean_, 'std': std_})

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
tl_labels = [r['label'] for r in tl_summary]
tl_means  = [r['mean']  for r in tl_summary]
tl_stds   = [r['std']   for r in tl_summary]
x_pos     = range(len(tl_labels))

ax.errorbar(x_pos, tl_means, yerr=tl_stds, marker='o', linewidth=2.5,
            markersize=10, capsize=6, color='#3498db',
            markerfacecolor='#2ecc71', markeredgecolor='navy',
            ecolor='#3498db', capthick=2)
ax.fill_between(x_pos,
                [m - s for m, s in zip(tl_means, tl_stds)],
                [m + s for m, s in zip(tl_means, tl_stds)],
                alpha=0.15, color='#3498db')
for i, (lbl, m) in enumerate(zip(tl_labels, tl_means)):
    ax.annotate(f'{m:.1f}%', (i, m), textcoords='offset points',
                xytext=(6, 9), fontsize=11, fontweight='bold')
ax.set_xticks(list(x_pos))
ax.set_xticklabels(tl_labels)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Transfer Learning: Accuracy vs. Target Adaptation Data',
             fontsize=13, fontweight='bold')
ax.set_ylim(50, 100); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/fig_transfer_learning.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: fig_transfer_learning.png")

## 🆕 Track 4: Freezing of Gait (FoG) Detection
Detect Freezing of Gait events in PD patients using EEG spectral features.
Requires FoG event markers in the dataset annotations (e.g., events.tsv).

In [ ]:
print("Checking for FoG annotations in dataset...")
fog_files = []
for fpath in eeg_files:
    if 'task-walk' not in fpath:
        continue
    events_file = fpath.replace('_eeg.set', '_events.tsv')
    if os.path.exists(events_file):
        try:
            df_ev = pd.read_csv(events_file, sep='\t')
            if 'trial_type' in df_ev.columns:
                has_fog = df_ev['trial_type'].astype(str).str.lower().str.contains(
                    'fog|freeze|freez').any()
                if has_fog:
                    fog_files.append(fpath)
        except Exception:
            pass

print(f"Files with FoG annotations: {len(fog_files)}")

if len(fog_files) >= 5:
    fog_records = []
    for fpath in fog_files:
        try:
            raw = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
            drop_ch = [ch for ch in raw.ch_names
                       if ch in ['VREF'] or 'EOG' in ch.upper()]
            raw.drop_channels([c for c in drop_ch if c in raw.ch_names])
            raw.filter(BANDPASS_LO, BANDPASS_HI, verbose=False)
            events_file = fpath.replace('_eeg.set', '_events.tsv')
            df_ev = pd.read_csv(events_file, sep='\t')
            subj_id = next((p for p in fpath.replace('\\', '/').split('/')
                            if p.startswith('sub-')), fpath)
            for _, row in df_ev.iterrows():
                onset_s = float(row['onset'])
                onset_s2 = int(onset_s * SFREQ)
                end_s    = onset_s2 + N_SAMPLES
                if end_s > raw.n_times:
                    continue
                seg = raw.get_data()[:, onset_s2:end_s]
                label_str = str(row.get('trial_type', '')).lower()
                is_fog = any(kw in label_str for kw in ['fog', 'freeze', 'freez'])
                fog_records.append({
                    'features': compute_psd_features(seg),
                    'label': int(is_fog),
                    'subject': subj_id
                })
        except Exception:
            pass

    if len(fog_records) >= 100:
        X_fog   = np.array([r['features'] for r in fog_records])
        y_fog   = np.array([r['label']    for r in fog_records])
        subs_fog = [r['subject']           for r in fog_records]
        unique_f = sorted(set(subs_fog))
        smap_f   = {s: i for i, s in enumerate(unique_f)}
        g_fog    = np.array([smap_f[s] for s in subs_fog])

        print(f"FoG epochs: {int(np.sum(y_fog==1))}, Normal walk: {int(np.sum(y_fog==0))}")
        res_fog = run_crossval(
            RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
            X_fog, y_fog, g_fog, 'FoG Detection (RF)',
            n_folds=min(N_FOLDS, len(unique_f)))
        print("\nFoG DETECTION RESULTS:")
        print(f"  Accuracy : {res_fog['Accuracy']}%")
        print(f"  F1 Score : {res_fog['F1 Score']}%")
    else:
        print("Not enough FoG epochs found.")
else:
    print("\nNo FoG markers found in ds007526.")
    print("ds007526 does not contain explicit Freezing of Gait labels.")
    print("To run FoG detection, use a dataset with FoG annotations, e.g.:")
    print("  - OpenNeuro ds003490 (Keren et al.)")
    print("  - PhysioNet MJFF Levodopa Response Study")
    print("\nThe FoG detection code above is ready — just change DATASET_PATH.")

## 📋 Section 9: Final Results Summary & CSV Export

In [ ]:
print("=" * 60)
print("       COMPLETE RESEARCH RESULTS SUMMARY")
print("=" * 60)

print("\n--- BASELINE BENCHMARK ---")
print(df_baseline[['Model', 'Accuracy', 'F1 Score', 'ROC-AUC']].to_string(index=False))

print("\n--- PD SEVERITY STRATIFICATION ---")
if len(severity_results) > 0:
    print(df_severity[['Group', 'N_Subjects', 'Accuracy']].to_string(index=False))
else:
    print("No severity results (insufficient subjects per group)")

print("\n--- BAND ABLATION ---")
print(df_ablation[['Condition', 'Accuracy']].to_string(index=False))

print("\n--- TRANSFER LEARNING ---")
for r in tl_summary:
    print(f"  {r['label']:<22}: {r['mean']:.2f} +/- {r['std']:.2f}%")

print("\n" + "=" * 60)
print("All experiments complete!")

# ── Export CSVs ──────────────────────────────────────────────────────────────
df_baseline.drop(columns=['acc_vals', 'acc_mean'], errors='ignore').to_csv(
    '/kaggle/working/results_baseline.csv', index=False)
if not df_severity.empty:
    df_severity.drop(columns=['acc_vals', 'acc_mean'], errors='ignore').to_csv(
        '/kaggle/working/results_severity.csv', index=False)
df_ablation.to_csv('/kaggle/working/results_band_ablation.csv', index=False)
pd.DataFrame(tl_summary).to_csv(
    '/kaggle/working/results_transfer_learning.csv', index=False)

print("\nOutput files saved to /kaggle/working/:")
print("  results_baseline.csv")
print("  results_severity.csv")
print("  results_band_ablation.csv")
print("  results_transfer_learning.csv")
print("  fig_severity_stratification.png")
print("  fig_band_ablation.png")
print("  fig_transfer_learning.png")